In [ ]:
!apt-get update && apt-get install -y ffmpeg
!pip install gradio yt-dlp "audio-separator[gpu]" soundfile numpy

In [ ]:
import os
import tempfile
import hashlib
import numpy as np
import soundfile as sf
import gradio as gr
import yt_dlp
from typing import Tuple
from audio_separator.separator import Separator

# ==========================================
# 1. YouTube Downloader Logic
# ==========================================
def youtube(url: str) -> str:
    if not url:
        raise gr.Error("Please input a YouTube URL")

    hash_str = hashlib.md5(url.encode()).hexdigest()
    tmp_file = os.path.join(tempfile.gettempdir(), f"{hash_str}")

    try:
        ydl_opts = {
            "format": "bestaudio/best",
            "outtmpl": tmp_file,
            "postprocessors": [
                {
                    "key": "FFmpegExtractAudio",
                    "preferredcodec": "mp3",
                    "preferredquality": "192",
                }
            ],
        }

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
    except Exception as e:
        print(e)
        raise gr.Error(f"Failed to download YouTube audio from {url}")

    return tmp_file + ".mp3"

# ==========================================
# 2. Separator Initialization
# ==========================================
separators = {
    "BS-RoFormer": Separator(output_dir=tempfile.gettempdir(), output_format="mp3"),
    "Mel-RoFormer": Separator(output_dir=tempfile.gettempdir(), output_format="mp3"),
    "HTDemucs-FT": Separator(output_dir=tempfile.gettempdir(), output_format="mp3"),
}

def load():
    separators["BS-RoFormer"].load_model("model_bs_roformer_ep_317_sdr_12.9755.ckpt")
    separators["Mel-RoFormer"].load_model("model_mel_band_roformer_ep_3005_sdr_11.4360.ckpt")
    separators["HTDemucs-FT"].load_model("htdemucs_ft.yaml")

# Retry logic for network blips
for _ in range(3):
    try:
        load()
        break
    except Exception as e:
        print(e)

# ==========================================
# 3. Audio Processing Logic
# ==========================================
def merge(outs):
    print(f"Merging {outs}")
    bgm = np.sum(np.array([sf.read(out)[0] for out in outs]), axis=0)
    print(f"Merged shape: {bgm.shape}")
    tmp_file = os.path.join(tempfile.gettempdir(), f"{outs[0].split('/')[-1]}_merged")
    sf.write(tmp_file + ".mp3", bgm, 44100)
    return tmp_file + ".mp3"

def separate(audio: str, model: str) -> Tuple[str, str]:
    separator = separators[model]
    outs = separator.separate(audio)
    outs = [os.path.join(tempfile.gettempdir(), out) for out in outs]

    # roformers return 2 files (vocals, instrumental)
    if len(outs) == 2:
        return outs[1], outs[0]

    # demucs returns 4 files, so we merge everything except vocals into the BGM
    if len(outs) == 4:
        bgm = merge(outs[:3])
        return outs[3], bgm

    raise gr.Error("Unknown output format")

def from_youtube(url: str, model: str) -> Tuple[str, str, str]:
    audio = youtube(url)
    return audio, *separate(audio, model)

# ==========================================
# 4. Gradio Interface (Lightweight)
# ==========================================
with gr.Blocks() as app:
    gr.Markdown("""
    # Vocal Separation SOTA 🎤 (Colab Edition)
    This is a demo for SOTA vocal separation models running natively in Google Colab.
    Upload an audio file or paste a YouTube link, and the model will separate the vocals from the background music.
    *(Spectrogram generation has been removed to support longer audio files without running out of RAM).*
    """)

    with gr.Row():
        with gr.Column():
            gr.Markdown("## Upload an audio file")
            audio = gr.Audio(label="Upload an audio file", type="filepath")
        with gr.Column():
            gr.Markdown("## or use a YouTube URL")
            yt = gr.Textbox(
                label="YouTube URL", placeholder="https://www.youtube.com/watch?v=..."
            )
            yt_btn = gr.Button("Use this YouTube URL")

    with gr.Row():
        model = gr.Radio(
            label="Select a model",
            choices=[s for s in separators.keys()],
            value="BS-RoFormer",
        )
        btn = gr.Button("Separate", variant="primary")

    with gr.Row():
        with gr.Column():
            vocals = gr.Audio(
                label="Vocals", format="mp3", type="filepath", interactive=False
            )
        with gr.Column():
            bgm = gr.Audio(
                label="Background", format="mp3", type="filepath", interactive=False
            )

    gr.Markdown(
        """
        ### Models
        - BS-RoFormer: https://arxiv.org/abs/2309.02612
        - Mel-RoFormer: https://arxiv.org/abs/2310.01809
        """
    )

    # Simplified event listeners (no .success chains for spectrograms)
    btn.click(
        fn=separate,
        inputs=[audio, model],
        outputs=[vocals, bgm],
        api_name="separate",
    )

    yt_btn.click(
        fn=from_youtube,
        inputs=[yt, model],
        outputs=[audio, vocals, bgm],
    )

app.launch(share=True, debug=True)